# CTRS Evaluation 

## 1. ตั้งค่า path และโหลด dialogues

In [1]:
import json
from pathlib import Path

BASE = Path(r"C:\Luna-AI-Therapist\dissonance")
DIALOG_DIR = BASE / "craft_dialogue"   # โฟลเดอร์แม่ของ dialogue_4,5,6

print("DIALOG_DIR =", DIALOG_DIR)

def load_dialogue_json(path: Path) -> str:
    """สำหรับไฟล์ .json ที่มี field 'turns'"""
    data = json.loads(path.read_text(encoding="utf-8"))
    lines = []
    for t in data["turns"]:
        lines.append(f"Client: {t['client']}")
        lines.append(f"Therapist: {t['therapist']}")
    return "\n".join(lines)

def load_dialogue_jsonl(path: Path) -> str:
    """สำหรับไฟล์ .jsonl: 1 turn ต่อ 1 บรรทัด"""
    lines = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            t = json.loads(line)
            # ปรับ key ให้ตรงกับไฟล์คุณ
            lines.append(f"Client: {t['client']}")
            lines.append(f"Therapist: {t['therapist']}")
    return "\n".join(lines)

def load_dialogue(path: Path) -> str:
    """เลือก loader ตามนามสกุลไฟล์"""
    if path.suffix == ".json":
        return load_dialogue_json(path)
    elif path.suffix == ".jsonl":
        return load_dialogue_jsonl(path)
    else:
        raise ValueError(f"Unsupported file type: {path}")

DIALOG_DIR = C:\Luna-AI-Therapist\dissonance\craft_dialogue


## 2. Set up CTRS

### Prepare CTRS mapping items 

In [2]:
CTRS_ITEMS = [
    {
        "name": "Agenda",
        "question": """(ใส่ Evaluation Question ของ Agenda จาก paper)""",
        "criteria": """(ใส่คำอธิบายเกณฑ์คะแนน 0–6 ของ Agenda)"""
    },
    {
        "name": "Feedback",
        "question": """(Evaluation Question ของ Feedback)""",
        "criteria": """(เกณฑ์คะแนน Feedback 0–6)"""
    },
    {
        "name": "Understanding",
        "question": """(Evaluation Question ของ Understanding)""",
        "criteria": """(เกณฑ์คะแนน Understanding 0–6)"""
    },
    {
        "name": "Interpersonal Effectiveness",
        "question": """(Evaluation Question ของ Interpersonal Effectiveness)""",
        "criteria": """(เกณฑ์คะแนน Interpersonal Effectiveness 0–6)"""
    },
    {
        "name": "Collaboration",
        "question": """(Evaluation Question ของ Collaboration)""",
        "criteria": """(เกณฑ์คะแนน Collaboration 0–6)"""
    },
    {
        "name": "Guided Discovery",
        "question": """(Evaluation Question ของ Guided Discovery)""",
        "criteria": """(เกณฑ์คะแนน Guided Discovery 0–6)"""
    },
    {
        "name": "Focus",
        "question": """(Evaluation Question ของ Focus)""",
        "criteria": """(เกณฑ์คะแนน Focus 0–6)"""
    },
    {
        "name": "Strategy",
        "question": """(Evaluation Question ของ Strategy)""",
        "criteria": """(เกณฑ์คะแนน Strategy 0–6)"""
    },
    # เพิ่ม item อื่น ๆ ตาม CTRS manual/paper ถ้ามี
]

### GPT 4o call function for evaluation each item

In [4]:
import getpass
from openai import OpenAI

# ใส่ key ครั้งเดียวต่อ kernel
OPENAI_API_KEY = getpass.getpass("OpenAI API key: ")
client = OpenAI(api_key=OPENAI_API_KEY)

SYSTEM_PROMPT = "You are a strict CBT supervisor scoring therapist behavior using the CTRS."

EVAL_TEMPLATE = """You are an evaluator. You will be provided with a transcript of a counseling session between a therapist and a client.
Your task is to assess the therapist based on the given CTRS item.

Please follow these steps:
1. Read the counseling session transcript carefully.
2. Read the evaluation question and the detailed criteria for scoring from 0 to 6.
3. Assign a single integer score from 0 to 6, grading very strictly.
   If there is any deficiency, however minor, assign a score of 4 or lower.
4. Output ONLY: "<score>, <short explanation>".

[Counseling conversation]
{conversation}

[Evaluation Question]
{question}

[Criteria]
{criteria}
"""

In [5]:
# Call llm function
def call_llm(user_prompt: str) -> str:
    """เรียก GPT-4o ส่งกลับข้อความเดียว (str)."""
    resp = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt},
        ],
        temperature=0.0,
    )
    return resp.choices[0].message.content.strip()

In [6]:
# Score_item function
def score_item(conversation: str, item: dict) -> tuple[int, str]:
    """ให้คะแนน CTRS 1 ข้อบน 1 dialogue."""
    user_prompt = EVAL_TEMPLATE.format(
        conversation=conversation,
        question=item["question"],
        criteria=item["criteria"],
    )
    text = call_llm(user_prompt)
    # คาดว่า LLM ตอบในรูปแบบ "4, brief explanation..."
    score_str, explanation = text.split(",", 1)
    return int(score_str.strip()), explanation.strip()

## 3. Run CTRS Evaluation

### Loop dialogues x CTRS items and save in dataframe

In [9]:
import pandas as pd

# หา .json / .jsonl ทุกไฟล์ใน subfolders (dialogue_4,5,6)
json_paths = list(DIALOG_DIR.rglob("*.json"))
jsonl_paths = list(DIALOG_DIR.rglob("*.jsonl"))

all_paths = sorted(json_paths + jsonl_paths)

print("Found JSON :", json_paths)
print("Found JSONL:", jsonl_paths)
print("Total files:", len(all_paths))

rows = []

for path in all_paths:
    dialogue_id = path.stem  # ชื่อไฟล์ไม่รวม .json/.jsonl

    # map เป็น condition ตามชื่อไฟล์
    name = path.name
    if "dialogue_4" in name:
        condition = "baseline"
    elif "dialogue_5" in name:
        condition = "emotion"
    elif "dialogue_6" in name:
        condition = "dissonance"
    else:
        condition = "unknown"

    print(f"\n=== Evaluating dialogue: {dialogue_id} ({condition}) ===")
    print("File:", path)

    # แปลงไฟล์ -> ข้อความบทสนทนา
    conversation = load_dialogue(path)

    # วนทุก CTRS item
    for item in CTRS_ITEMS:
        item_name = item["name"]
        print(f"- Scoring CTRS item: {item_name} ...", end=" ")

        try:
            score, explanation = score_item(conversation, item)
        except Exception as e:
            print("ERROR:", e)
            score, explanation = None, f"ERROR: {e}"
        else:
            print(f"score = {score}")

        rows.append({
            "dialogue_id": dialogue_id,
            "condition": condition,
            "file_path": str(path),
            "item": item_name,
            "score": score,
            "explanation": explanation,
        })

# รวมผลเป็น DataFrame
df_ctrs = pd.DataFrame(rows)
print("Total rows:", len(df_ctrs))
df_ctrs.head()

Found JSON : [WindowsPath('C:/Luna-AI-Therapist/dissonance/craft_dialogue/dialogue_6/dialogue_6_full_dissonance_online.json')]
Found JSONL: [WindowsPath('C:/Luna-AI-Therapist/dissonance/craft_dialogue/dialogue_4/dialogue_4_full_baseline.jsonl'), WindowsPath('C:/Luna-AI-Therapist/dissonance/craft_dialogue/dialogue_5/dialogue_5_full_emotion_online.jsonl'), WindowsPath('C:/Luna-AI-Therapist/dissonance/craft_dialogue/dialogue_6/dialogue_6_full_dissonance_online.jsonl')]
Total files: 4

=== Evaluating dialogue: dialogue_4_full_baseline (baseline) ===
File: C:\Luna-AI-Therapist\dissonance\craft_dialogue\dialogue_4\dialogue_4_full_baseline.jsonl
- Scoring CTRS item: Agenda ... ERROR: invalid literal for int() with base 10: "I'm sorry"
- Scoring CTRS item: Feedback ... score = 4
- Scoring CTRS item: Understanding ... score = 4
- Scoring CTRS item: Interpersonal Effectiveness ... score = 4
- Scoring CTRS item: Collaboration ... score = 4
- Scoring CTRS item: Guided Discovery ... score = 0
- Sco

TypeError: list indices must be integers or slices, not str

### Generate summary CTRS table

In [ ]:
df_summary = (
    df_ctrs
    .groupby(["condition", "item"])["score"]
    .mean()
    .reset_index()
    .pivot(index="item", columns="condition", values="score")
)
df_summary

KeyError: 'condition'

### Export in `csv` file

In [ ]:
summary.to_csv(BASE / "evaluation" / "counselingeval_ctrs_table.csv")